In [1]:
import numpy as np
import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import statistics
from pathlib import Path
import re
import requests
from io import StringIO
import random
import pdb
from io import BytesIO
import geopandas as gpd
import openpyxl
import zipfile
import requests

In this code, firstly we incorporate two datasets to obtain a final dataset including the land uses of cadastral references (RC) in Gipuzkoa. Later, we incorporate data related to incomes for each RC.

# Bring land use data from Gipuzkoa Irekia, related to establishments
Bienes Inmuebles de Naturaleza Urbana existentes en el Catastro de Gipuzkoa

https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/detail/detailView/1d7a519e-5248-4e87-85f7-ecf3d00a1e52

In [2]:
# List with all the urls to local data of the ~90 towns in the analysis

urls_locales=[
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/55867f8d-1799-4a95-a7c0-36275c1ed1cf', # Abaltzisketa
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/a23edbb4-e6a3-48b7-bc60-c7df89f79ae7', # Aduna
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/afc3353c-d009-4269-b79a-7788567d7c7e', # Aia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/bcb0e042-bf3a-4b5f-a3f0-5fb3bb144748', # Aizarnazabal
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/79e66092-0f2b-4f28-94fb-e93362d0066e', # Albiztur
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/172150d2-7b28-4c42-9626-700c20e90585', # Alegia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/7c44c34c-852a-4fc7-b711-121a08bae4ec', # Alkiza
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/297c041e-638a-4123-a53c-6504dccb5c48', # Altzaga
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/362a2b37-9945-4ed3-b003-e6a15013b517', # Altzo
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/3f75d895-94bd-407b-82d7-0c841829932d', # Amezketa
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/c6abbdeb-2583-420d-b6f1-46f37874dbf9', # Andoain
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/242280af-e62c-498f-a40d-cd33606abb80', # Anoeta
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/f5128ccd-ba6f-465f-8b52-34c7decfee90', # Antzuola
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/b336dfa5-1362-4962-ba82-bfa43238c043', # Arama
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/38f8aa02-307d-48d8-a5fa-eed285d13180', # Aretxabaleta
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/b36a2241-b4c4-45fb-8ce4-523c9716741a', # Arrasate
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/0f49a7a4-d203-438e-acb3-dadaf3b50984', # Asteasu
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/c0ee5e2b-71ce-4c89-8ba9-7a9ef7f4b1b5', # Astigarraga
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/4f8d4b25-e89d-48fd-a7cf-9c368f971898', # Ataun
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/c255519e-d335-406a-be8d-8f2e469ba3f9', # Azkoitia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/32865b0e-e05e-4b87-8211-61e8526aba6c', # Azpeitia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/88a9a6ea-3089-4c10-8586-d49b508ba18a', # Baliarrain
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/873c1579-3893-4e44-a221-9a399465593f', # Beasain
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/616b11c7-8b66-4e31-96ce-67487d343835', # Beizama
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/e3614d03-07b2-481d-9d59-38cab4e066a8', # Belauntza
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/2d2f53df-0b71-4e18-a147-08a26f821013', # Berastegi
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/876d7968-d6b2-4b8c-bbae-899bb35444f6', # Bergara
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/e916007c-5a49-44af-a289-aa8951c25c9d', # Berrobi
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/02c6bef7-88d5-45dd-bfc4-eb68ce3d6987', # Bidania-Goiatz
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/f423a594-00e7-4f4c-9ebf-f3ff352cef51', # Deba
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/1bd56664-f2b7-42b0-ba25-e54b3d960726', # Eibar
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/0b6e1195-1299-4256-b261-34ef9a9cacd7', # Elduain
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/6dcadbad-add9-4c98-967f-740bbe9641f7', # Elgeta
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/4a807b4f-1302-4384-b1fb-bf420ff9f03d', # Elgoibar
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/1ddf5167-6159-4f7a-b43f-c6e7e92225ba', # Errenteria
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/17c8497d-1388-47fd-a46e-6727f9a21641', # Errezil
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/596e07a6-e00d-47b9-8f1f-60d442f2af46', # Eskoriatza
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/07182ac8-675d-499e-b06f-e77c5633cd7e', # Ezkio
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/02617b0e-646c-47b0-b5b4-f96d853a86c9', # Gabiria
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/385584b5-0df7-46be-a77c-c983681a7c30', # Gaintza
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/2a126f2e-f8df-4656-b368-b98b0db75277', # Gaztelu
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/8f755979-9408-4c61-b354-7fcc50ad4577', # Getaria
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/313dc81d-0f1a-4cd8-82e6-d81ae90e5019', # Hernani
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/09a6cc04-5626-41e4-9261-7ffc0ad80cd8', # Hernialde
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/12088aca-f753-4dd7-814d-4ac5506c41e7', # Hondarribia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/85832021-c74c-4dd0-b516-0c32e84d80b8', # Ibarra
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/cf66475a-480b-44cc-b3a3-e8621d656ed3', # Idiazabal
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/397e33b1-3acf-4fe6-bdc3-8ee928808710', # Ikaztegieta
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/9c6c71ae-d062-4fa2-b28e-55a9ba9f8de9', # Irun
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/0cc56ba6-47a5-40a8-ad69-bfa20a8dedc8', # Irura
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/6f38056a-5442-4609-8871-436dbaa295c3', # Itsasondo
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/418e5c7a-a76c-48af-9be1-ed0b90547313', # Larraul
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/70dc4dae-d5ed-4397-8ef0-d56f46060b33', # Lasarte-Oria
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/2722f0eb-7ef2-4c4d-baca-454353f5caed', # Lazkao
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/cf455803-cbe6-4fb8-95a3-c932b8c9ff4e', # Leaburu-Txarama
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/ff1c3ce6-bb6b-4211-9f76-2faacde64ce7', # Legazpi
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/50833eac-6fd3-4219-bc47-d563cd17a0b6', # Legorreta
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/e99e13bf-0e32-4bf0-a9b7-7733b680cda8', # Leintz-Gatzaga
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/da583fca-965b-4832-ad8e-680a6f98b9c6', # Lizartza
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/c1ef8ad8-655a-48fd-8d76-44964f265011', # Lezo
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/189994ee-99bc-439b-8926-88f90b450b41', # Mendaro
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/d9bdb912-7a16-43d7-88b7-67c2395cdda8', # Mutiloa
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/7a32f635-e606-44f2-bf8e-3f12468a88d2', # Mutriku
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/080bfc9b-1909-4832-a428-e92c32cffa2d', # Oiartzun
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/cd7b91d8-65f9-41f5-a2b9-5bac70d9b884', # Olaberria
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/6498dbc2-71c8-4629-9103-9abafe6374b2', # Ordizia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/14572817-dca2-4e60-907f-2605017dc2d2', # Orendain
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/1000acac-8a36-4a8c-8149-fb9a90a4d2ba', # Orexa
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/cdf57eda-5acc-4662-ad98-32c516352027', # Orio
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/da61cb4f-889b-444d-bdc8-b6d4251ec1ee', # Ormaiztegi
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/24a8f343-293d-426c-b6ab-7d66e75b8d58', # Oñati
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/6822ee92-e76f-4f53-b192-fbd102fb7763', # Pasaia
      # '', # San Sebastián 1
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/6be2d256-27ca-4de5-9de9-fae69243f8d8', # Segura
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/7903f1b2-7ae7-443d-b7d0-84f0958a4c0e', # Soraluze
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/de6fa074-0792-4951-89c7-e08d4eae5340', # Tolosa
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/805740c6-e11f-43b8-9dbd-7edc994da6b7', # Urnieta
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/fbdd7aa7-f1c1-4149-bc74-3995a7c13c2f', # Urretxu
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/03bfc3a4-c0d4-491e-a163-ef983aa0403a', # Usurbil
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/8773c064-1976-409f-a355-7847f8420d09', # Villabona
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/15cca386-ada3-4340-8196-8a44720c3adc', # Zaldibia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/62434408-12b5-4a7a-84ba-22d5ca426a72', # Zarautz
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/72402b02-973f-4928-84ff-c70263a9eab8', # Zegama
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/969a0430-f78f-475a-b329-399ca36a60ed', # Zerain
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/7c86135e-b5fa-4314-ba48-ed1b3297a446', # Zestoa
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/4a6cb882-5ba2-4b21-bc50-9571a0f2649c', # Zizurkil
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/e4ab4c6e-af2d-4c9d-b632-8e933193cef2', # Zumaia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/0fa7d40a-81f4-49c3-9414-182ab6fc70e3', # Zumarraga
      ]
len(urls_locales)

# List with all the urls to rural data of the ~90 towns in the analysis

urls_parcelas=[
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/278059f2-07b1-4e10-9fc3-484c04e8567d', # Abaltzisketa
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/2d1e98ec-42ad-4377-9d9e-fea6ed6a6c88', # Aduna
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/49bc9c3a-48e3-4fe8-950a-a614c3340c44', # Aia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/b7d83d31-4376-4061-8330-0138cb7ec0c1', # Aizarnazabal
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/4d14f55c-cad2-46b6-84c0-f966edd4623b', # Albiztur
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/53ec54d0-6615-4273-b768-6ab5102bd9fd', # Alegia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/0ef397e5-1b09-4167-a64e-2b1b888f3ea7', # Alkiza
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/9aaccf3b-bf04-4d27-8e3f-6159e5f40708', # Altzaga
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/a1228142-d6bb-4fe0-b5e4-e33a8be703dd', # Altzo
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/b845af39-3655-42fa-b21a-f787f5bbc15c', # Amezketa
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/5cef5d53-c4e7-42c5-a389-2546bf5ad7f1', # Andoain
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/5fc9717a-5a02-4f85-9b15-c88c2ef47c06', # Anoeta
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/6403ee6f-7768-4ca9-90f9-b914ee57a6d7', # Antzuola
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/3679d98a-a1f6-4c18-ad4b-7445d5d032ab', # Arama
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/ffb817af-1ce2-4cc1-95bc-9259fd40310a', # Aretxabaleta
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/7a020311-a3f1-4085-99fb-572180289d3f', # Arrasate
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/ad5d489c-1d35-4cc7-a57d-e4f0e9c22469', # Asteasu
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/32691844-b74c-4de2-8b1a-25cd4853581d', # Astigarraga
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/7bbbb1ff-166f-4d51-97db-e7a25d8fe845', # Ataun
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/91e48cc6-5959-469d-a1c4-a0752b363d39', # Azkoitia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/c344d2fa-0af1-4220-9af1-6aa6f7f60006', # Azpeitia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/c4020963-8c41-4315-9112-ba465dc02450', # Baliarrain
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/b3c22d55-c1a6-40ab-9c6a-4b80bc56ddb8', # Beasain
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/ead01c3d-d200-411d-aa5a-5a2a828fc2fd', # Beizama
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/04704a63-7abd-4d5d-860b-90f2359f8048', # Belauntza
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/6986b435-9ab9-4f75-b095-9fae92ed962d', # Berastegi
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/35bc3689-6167-496f-82be-3cf5d4aa0b76', # Bergara
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/31ec956e-57b1-450c-9a8b-b6d179326c52', # Berrobi
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/15133bfd-8338-4b84-b2f0-8b1506c829fc', # Bidania-Goiatz
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/3b8688eb-ba41-4f93-b1fe-131f657eaa15', # Deba
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/b2d945d8-ce38-4f48-83fb-89d213836113', # Eibar
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/473ed2b0-86d5-471c-8b4c-c0dea0399d51', # Elduain
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/53dcf14e-58ac-4b07-9d6e-338247cfb4e2', # Elgeta
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/de96fff2-b524-4411-9825-58798222c08e', # Elgoibar
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/18e1d369-0feb-4e17-91d5-bc974fb3c449', # Errenteria
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/a46358f3-1474-4339-a3f8-e2a7e20ac3e5', # Errezil
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/ed2b2ee5-b069-458f-9d06-51f7068d4343', # Eskoriatza
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/cd43f187-d045-4935-954a-7ab297137dcd', # Ezkio
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/16c5b4d8-2b2d-4523-bca1-5582a286006c', # Gabiria
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/a97ab87f-1561-4d1e-847c-8b969e67a033', # Gaintza
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/a5dbb9fd-c4fe-4d0e-a961-fb7ebdff272e', # Gaztelu
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/3ab940e2-a848-47c1-80f4-cd59c3fad5b0', # Getaria
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/e0c4318c-3fa7-4491-81ce-6da265238922', # Hernani
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/75490367-935a-4465-83f5-3ed2a974cb98', # Hernialde
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/5f27183d-1eaf-4c4e-9612-ff21ff30c4a2', # Hondarribia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/fb65c039-560e-4068-823e-67dade76e7eb', # Ibarra
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/9aae1998-8869-43cc-bc32-713de9fcb31c', # Idiazabal
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/d5139076-91c2-4442-a499-36a5227141ff', # Ikaztegieta
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/d4714cbe-8d10-43bf-b5d3-df11e9c7f1fa', # Irun
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/b61e324c-4f70-4bb4-a7cb-6e056c3df03e', # Irura
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/5789b4b2-a8f3-439e-aeb3-2fa73fcd214c', # Itsasondo
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/e9796773-c90e-4be6-a830-f114fa7ca790', # Larraul
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/817d4351-25b4-47df-b542-43d6fe18071a', # Lasarte-Oria
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/0115c1d7-f696-4d19-8ffa-089c715ff13a', # Lazkao
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/d8e30173-f5ee-4481-9f26-99701fb5eec8', # Leaburu-Txarama
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/9ecab781-0a42-4784-95af-4a702c583e55', # Legazpi
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/71b4b450-fe04-4c68-ae67-8819be8695cf', # Legorreta
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/e70939d7-1e20-4438-9483-60edc1202bb7', # Leintz-Gatzaga
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/cfca06ba-5cbe-4a26-95e0-26b71855e6f0', # Lizartza
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/154eafc7-b7d2-4560-aaee-1315aa8f17bd', # Lezo
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/332aa440-05b6-4f7a-8f68-6fcb84109cb1', # Mendaro
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/4b14688e-af10-41ae-bed0-91e867b82dc2', # Mutiloa
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/8474ba1d-6c1e-440e-ba96-4cf2147e78fd', # Mutriku
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/7ec86b32-d39d-4a74-ab8a-58c8cb2f2fc6', # Oiartzun
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/4151adf8-0352-4232-a591-eb5645961913', # Olaberria
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/dbe6ff79-844f-4857-a194-dbc1f0198131', # Ordizia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/8f286215-8a56-4d2b-832e-9a19e988adf8', # Orendain
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/f446c156-bcbb-4d16-afc0-41bbd649eb31', # Orexa
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/e84df13e-7ca3-4b7c-9341-ef44143e4a2e', # Orio
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/6f543a6b-6228-4e7b-81c7-3cda98109b16', # Ormaiztegi
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/acc838ee-1317-4f26-b881-38600cb5ca98', # Oñati
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/4c53d92d-57f3-407a-bc15-bbdb731f904d', # Pasaia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/a678f769-01fd-4f25-b949-05a3d1f68286', # San Sebastián
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/4e7e673d-d1dd-4ffd-acb8-e8cd74ece65b', # Segura
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/66e7a3b1-1c18-4f4f-bc59-2cda49671ec0', # Soraluze
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/30368f7b-94d4-4973-a765-03a4d898e9d7', # Tolosa
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/5c523027-1cf0-420f-909e-7fbc29e8ebf2', # Urnieta
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/17f1a071-fe3d-4625-857d-373acfc6d34a', # Urretxu
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/988d6a26-1a30-4ebc-8895-65f70ec303e1', # Usurbil
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/ff80b6d3-2636-4cde-a152-e79fb22afb9b', # Villabona
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/f4bd6a47-a38c-4823-a0c7-7a28dad1e526', # Zaldibia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/07ff38a9-c25e-4617-8e3d-cd8239de9ef3', # Zarautz
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/85c4dfb5-ee78-4fed-8865-fc07676657ea', # Zegama
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/b33f77a6-6ea9-488c-9f10-9c84ffb64382', # Zerain
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/bbb25303-fd50-44aa-93d6-0a5563fccb5f', # Zestoa
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/10d36bba-9919-49bb-8d86-be2b919c6732', # Zizurkil
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/162fd60c-94ec-40ee-8ca8-e3effc54b807', # Zumaia
      'https://www.gipuzkoairekia.eus/es/datu-irekien-katalogoa/-/openDataSearcher/download/downloadResource/b8dc4fd8-bdbd-4b5b-a32c-30bbf337d309', # Zumarraga
      ]
len(urls_parcelas)

88

In [3]:
# Donosti has a unique url but 4 files, I download them manually

locales_donosti_df = []

locales_donosti = ["locales_donostia/ficherourbanalocalesmunicipio69.csv",
                   "locales_donostia/ficherourbanalocalesmunicipio69-2.csv",
                   "locales_donostia/ficherourbanalocalesmunicipio69-3.csv",
                   "locales_donostia/ficherourbanalocalesmunicipio69-4.csv"]

for path in locales_donosti:
    df = gpd.read_file(path, encoding='latin1')
    locales_donosti_df.append(df)
locales_donosti_df = pd.concat(locales_donosti_df, ignore_index=True)
locales_donosti_df['Mun'] = locales_donosti_df['Mun'].astype(int)
locales_donosti_df

,Mun,Referen,N.Fijo,D,Pal,Nlo,Nuc,Cvia,Descripción Vía,Npor,Es,Pl,Man,De,Superfic.,Om
0,69,7494001,6036420,V,P,001,001,3590,"PADRE ORKOLAGA, PASEO DEL",200,,00,,V,"153,50",EU
1,69,7494001,6036420,V,,002,002,3590,"PADRE ORKOLAGA, PASEO DEL",200,,00,,B,"53,29",EC
2,69,7494001,6036420,V,,003,002,3590,"PADRE ORKOLAGA, PASEO DEL",200,,00,,B,"53,29",EC
3,69,7494001,6036420,V,,004,002,3590,"PADRE ORKOLAGA, PASEO DEL",200,,00,,B,"147,00",EC
4,69,7494002,6049056,F,P,001,001,3590,"PADRE ORKOLAGA, PASEO DEL",204,,00,,V,"113,27",EU
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
234300,69,8991007,3071409,C,,005,003,6170,"BENTATXIKI, CAMINO DE",006,,00,,B,"80,00",MI
234301,69,8991007,3071409,C,,006,004,6170,"BENTATXIKI, CAMINO DE",006,,00,,T,"50,00",MI
234302,69,8991007,3071409,C,,007,005,6170,"BENTATXIKI, CAMINO DE",006,,00,,T,"100,00",MI
234303,69,8991007,3071409,C,,008,006,6170,"BENTATXIKI, CAMINO DE",006,,00,,B,"120,00",MI


In [4]:
# I generate a dataframe with all the data of the 88 towns in the analysis, and I add Donosti's data to it
# There are 939k estblishments in total, 234k of them in Donosti

locales = pd.DataFrame()

for url in urls_locales:
  response = requests.get(url)
  content = response.content.decode('ISO-8859-1')
  csv_data = BytesIO(content.encode('utf-8'))
  df_provisional=pd.read_csv(csv_data, sep=";", index_col=False, low_memory=False)
  locales = pd.concat([locales, df_provisional], ignore_index=True)
locales.rename(columns={'Descripción Vía               ': 'Descripción Vía', 'N.Fijo ': 'N.Fijo'}, inplace=True)

# Add Donosti's locals to the rest
locales = pd.concat([locales, locales_donosti_df], ignore_index=True)

# Tratamiento
locales['Mun'] = pd.to_numeric(locales['Mun'], errors='coerce')
locales = locales.dropna(subset=['Mun'], axis=0)
locales['Mun'] = locales['Mun'].astype(int)
locales['Superfic.'] = locales['Superfic.'].str.replace(',', '.')
locales['Superfic.'] = locales['Superfic.'].astype(float)

# From symbol to real use
usos = {
    'De': ['A ','B ','C ','D ','E ','G ','H ','I ','J ','K ','L ','M ','N ','O ','P ','Q ','R ','S ','T ','U ','V ','W ','X ','Y ','Z '],
    'Destino': ['Cultural', 'Agropecuario', 'Local Comercial','Deposito','Educativo','Garaje Cerrado','Hosteleria','Industrial','Jardin','Deportivo','Edificabilidad','Almacen','Sanidad','Oficina','Plaza Parking','Altura menor 1.5 mts','Religioso','Solar','Trastero','Obras Urbanizacion','Vivienda','Accesos y pasillos','Duplex','Espectaculos','Terraza']
}
usos = pd.DataFrame(usos)
mapeo = dict(zip(usos['De'], usos['Destino']))
locales['De'] = locales['De'].replace(mapeo)
locales.rename(columns={'Referen': 'Referencia'}, inplace=True)
locales=locales.reset_index()
locales=locales.drop(columns='index')
locales['Superfic.'] = locales['Superfic.'].astype(int)

locales

,Mun,Referencia,N.Fijo,D,Pal,Nlo,Nuc,Cvia,Descripción Vía,Npor,Es,Pl,Man,De,Superfic.,Om
0,1,7166001,5196877,R,P,1,1,6009,IRIONDO-ALTUN,007,,01,,Vivienda,120,MP
1,1,7166001,5196877,R,,2,1,6009,IRIONDO-ALTUN,007,,00,,Agropecuario,192,MP
2,1,7166001,5196877,R,,3,1,6009,IRIONDO-ALTUN,007,,01,,Agropecuario,72,MP
3,1,7166001,5196877,R,,4,1,6009,IRIONDO-ALTUN,007,,02,,Agropecuario,192,MP
4,1,7166002,5179123,D,,1,1,6009,IRIONDO-ALTUN,009,,00,,Garaje Cerrado,32,EC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
939416,69,8991007,3071409,C,,005,003,6170,"BENTATXIKI, CAMINO DE",006,,00,,Agropecuario,80,MI
939417,69,8991007,3071409,C,,006,004,6170,"BENTATXIKI, CAMINO DE",006,,00,,Trastero,50,MI
939418,69,8991007,3071409,C,,007,005,6170,"BENTATXIKI, CAMINO DE",006,,00,,Trastero,100,MI
939419,69,8991007,3071409,C,,008,006,6170,"BENTATXIKI, CAMINO DE",006,,00,,Agropecuario,120,MI


In [5]:
# Group locals (establishments) within the same cadastral reference and calculate land-use percentages
# 58k cadastral references in Gipuzkoa

def custom_apply(group):
    destinos = []
    porcentajes = []
    superficie_total = {}


    for destino, area in zip(group['De'], group['Superfic.']):
        if destino not in destinos:
            destinos.append(destino)
            porcentajes.append(0)
            superficie_total[destino] = 0

        index = destinos.index(destino)
        porcentajes[index] += (area / group['Superfic.'].sum()) * 100
        superficie_total[destino] += area

    return (destinos, porcentajes, superficie_total)

grouped = locales.groupby('Referencia').apply(custom_apply)

# Create new df
analisis_x_referencia = {'Referencia': [], 'Destinos': [], 'Porcentajes': [], 'Superficie Total': []}
for referencia, (destinos, porcentajes, superficie_total) in grouped.items():
    analisis_x_referencia['Referencia'].append(referencia)
    analisis_x_referencia['Destinos'].append(destinos)
    analisis_x_referencia['Porcentajes'].append([round(p, 2) for p in porcentajes])
    analisis_x_referencia['Superficie Total'].append([superficie_total[d] for d in destinos])

analisis_x_referencia = pd.DataFrame(analisis_x_referencia)

# Add town code to analisis_x_referencia
columnas=['Mun','Referencia']
mun_referencia=locales[columnas]
mun_referencia = mun_referencia.drop_duplicates(subset=['Referencia'])
mun_referencia = mun_referencia.reset_index()
mun_referencia = mun_referencia.drop(columns='index')

analisis_x_referencia = analisis_x_referencia.merge(mun_referencia, on='Referencia', how='left')

analisis_x_referencia.to_csv("analisis_x_referencia.csv", index=False)

analisis_x_referencia

C:\Users\iazcarateu\AppData\Local\Temp\ipykernel_16048\2841708234.py:17: RuntimeWarning: invalid value encountered in scalar divide
  porcentajes[index] += (area / group['Superfic.'].sum()) * 100
C:\Users\iazcarateu\AppData\Local\Temp\ipykernel_16048\2841708234.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = locales.groupby('Referencia').apply(custom_apply)


,Referencia,Destinos,Porcentajes,Superficie Total,Mun
0,3361001,"[Vivienda, Agropecuario]","[15.99, 84.01]","[126, 662]",34
1,3361002,"[Vivienda, Garaje Cerrado, Trastero]","[54.76, 23.02, 22.22]","[138, 58, 56]",34
2,3361003,"[Vivienda, Agropecuario]","[27.61, 72.39]","[135, 354]",34
3,3361004,"[Vivienda, Garaje Cerrado, Trastero]","[57.42, 20.1, 22.49]","[240, 84, 94]",34
4,3361005,"[Vivienda, Agropecuario, Garaje Cerrado, Trast...","[54.22, 36.72, 6.61, 2.45]","[443, 300, 54, 20]",34
...,...,...,...,...,...
58680,9999261,[Solar],[100.0],[3253],45
58681,9999262,[Solar],[100.0],[257],45
58682,9999263,[Solar],[100.0],[98],45
58683,9999264,[Solar],[100.0],[131],45


In [6]:
analisis_x_referencia = pd.read_csv("analisis_x_referencia.csv")

# Bring shapefiles with buildings

We are uploading the geometry and geolocation of each cadastral reference (RC) from catastro. A RC can be seen as a "building".

In both the previous analysis (establishments-locals) and now we are obtaining 58k RCs, it's consistent.

From "Catastro" webpage: https://ssl6.gipuzkoa.eus/CATASTRO/map.htm?id=74&idioma=esp

Many times, each RC is a unique building and inside the building there are locals with different land uses

Other times, a RC is many buildings, and each building has locals with dif. land uses

In [ ]:
# This in an auxiliary code to unzip all the files downloaded from Gipuzkoa's open data portal, which are in zip format. I use it to unzip the files of the 88 towns in the analysis, which I downloaded manually.

def unzip_all(zip_folder: str):
    # Define the output folder
    unzipped_folder = os.path.join(zip_folder, "unzipped")
    os.makedirs(unzipped_folder, exist_ok=True)

    # Iterate through all files in the folder
    for file in os.listdir(zip_folder):
        file_path = os.path.join(zip_folder, file)
        
        # Check if it's a zip file
        if zipfile.is_zipfile(file_path):
            with zipfile.ZipFile(file_path, 'r') as zip_ref:
                # Extract to a new subfolder inside "unzipped"
                extract_path = os.path.join(unzipped_folder, os.path.splitext(file)[0])
                os.makedirs(extract_path, exist_ok=True)  # Ensure folder exists
                zip_ref.extractall(extract_path)
                print(f"Extracted {file} to {extract_path}")

    print("All files have been unzipped.")

# Example usage:
unzip_all("C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro")

Extracted SHP-A001.zip to C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro\unzipped\SHP-A001
Extracted SHP-A002.zip to C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro\unzipped\SHP-A002
Extracted SHP-A003.zip to C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro\unzipped\SHP-A003
Extracted SHP-A004.zip to C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro\unzipped\SHP-A004
Extracted SHP-A005.zip to C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro\unzipped\SHP-A005
Extracted SHP-A006.zip to C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro\unzipped\SHP-A006
Extracted SHP-A007.zip to C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro\unzipped\SHP-A007
Extracted SHP-A008.zip to C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro\unzipped\SHP-A008
Extracted SHP-A009.zip to C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro\unzipped\SHP-A009
Extracted SHP-A010.zip to C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro\unzipped\SHP-A010


In [8]:
buildings_catastro = []



paths_shapefiles=["C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A001/001_HIRILUR_SHP/PARTZELAK-A001.shp",
                  "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A002/002_HIRILUR_SHP/PARTZELAK-A002.shp",
                  "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A003/003_HIRILUR_SHP/PARTZELAK-A003.shp",
                  "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A004/004_HIRILUR_SHP/PARTZELAK-A004.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A005/005_HIRILUR_SHP/PARTZELAK-A005.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A006/006_HIRILUR_SHP/PARTZELAK-A006.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A007/007_HIRILUR_SHP/PARTZELAK-A007.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A008/008_HIRILUR_SHP/PARTZELAK-A008.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A009/009_HIRILUR_SHP/PARTZELAK-A009.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A010/010_HIRILUR_SHP/PARTZELAK-A010.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A011/011_HIRILUR_SHP/PARTZELAK-A011.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A012/012_HIRILUR_SHP/PARTZELAK-A012.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A013/013_HIRILUR_SHP/PARTZELAK-A013.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A014/014_HIRILUR_SHP/PARTZELAK-A014.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A015/015_HIRILUR_SHP/PARTZELAK-A015.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A016/016_HIRILUR_SHP/PARTZELAK-A016.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A017/017_HIRILUR_SHP/PARTZELAK-A017.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A018/018_HIRILUR_SHP/PARTZELAK-A018.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A019/019_HIRILUR_SHP/PARTZELAK-A019.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A020/020_HIRILUR_SHP/PARTZELAK-A020.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A021/021_HIRILUR_SHP/PARTZELAK-A021.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A022/022_HIRILUR_SHP/PARTZELAK-A022.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A023/023_HIRILUR_SHP/PARTZELAK-A023.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A024/024_HIRILUR_SHP/PARTZELAK-A024.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A025/025_HIRILUR_SHP/PARTZELAK-A025.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A026/026_HIRILUR_SHP/PARTZELAK-A026.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A027/027_HIRILUR_SHP/PARTZELAK-A027.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A028/028_HIRILUR_SHP/PARTZELAK-A028.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A029/029_HIRILUR_SHP/PARTZELAK-A029.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A030/030_HIRILUR_SHP/PARTZELAK-A030.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A031/031_HIRILUR_SHP/PARTZELAK-A031.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A032/032_HIRILUR_SHP/PARTZELAK-A032.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A033/033_HIRILUR_SHP/PARTZELAK-A033.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A034/034_HIRILUR_SHP/PARTZELAK-A034.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A035/035_HIRILUR_SHP/PARTZELAK-A035.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A036/036_HIRILUR_SHP/PARTZELAK-A036.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A037/037_HIRILUR_SHP/PARTZELAK-A037.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A038/038_HIRILUR_SHP/PARTZELAK-A038.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A039/039_HIRILUR_SHP/PARTZELAK-A039.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A040/040_HIRILUR_SHP/PARTZELAK-A040.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A041/041_HIRILUR_SHP/PARTZELAK-A041.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A042/042_HIRILUR_SHP/PARTZELAK-A042.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A043/043_HIRILUR_SHP/PARTZELAK-A043.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A044/044_HIRILUR_SHP/PARTZELAK-A044.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A045/045_HIRILUR_SHP/PARTZELAK-A045.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A046/046_HIRILUR_SHP/PARTZELAK-A046.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A047/047_HIRILUR_SHP/PARTZELAK-A047.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A048/048_HIRILUR_SHP/PARTZELAK-A048.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A049/049_HIRILUR_SHP/PARTZELAK-A049.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A050/050_HIRILUR_SHP/PARTZELAK-A050.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A051/051_HIRILUR_SHP/PARTZELAK-A051.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A052/052_HIRILUR_SHP/PARTZELAK-A052.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A053/053_HIRILUR_SHP/PARTZELAK-A053.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A054/054_HIRILUR_SHP/PARTZELAK-A054.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A055/055_HIRILUR_SHP/PARTZELAK-A055.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A056/056_HIRILUR_SHP/PARTZELAK-A056.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A057/057_HIRILUR_SHP/PARTZELAK-A057.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A058/058_HIRILUR_SHP/PARTZELAK-A058.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A059/059_HIRILUR_SHP/PARTZELAK-A059.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A060/060_HIRILUR_SHP/PARTZELAK-A060.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A061/061_HIRILUR_SHP/PARTZELAK-A061.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A062/062_HIRILUR_SHP/PARTZELAK-A062.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A063/063_HIRILUR_SHP/PARTZELAK-A063.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A064/064_HIRILUR_SHP/PARTZELAK-A064.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A065/065_HIRILUR_SHP/PARTZELAK-A065.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A066/066_HIRILUR_SHP/PARTZELAK-A066.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A067/067_HIRILUR_SHP/PARTZELAK-A067.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A068/068_HIRILUR_SHP/PARTZELAK-A068.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A069/069_HIRILUR_SHP/PARTZELAK-A069.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A070/070_HIRILUR_SHP/PARTZELAK-A070.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A071/071_HIRILUR_SHP/PARTZELAK-A071.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A072/072_HIRILUR_SHP/PARTZELAK-A072.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A073/073_HIRILUR_SHP/PARTZELAK-A073.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A074/074_HIRILUR_SHP/PARTZELAK-A074.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A075/075_HIRILUR_SHP/PARTZELAK-A075.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A076/076_HIRILUR_SHP/PARTZELAK-A076.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A077/077_HIRILUR_SHP/PARTZELAK-A077.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A078/078_HIRILUR_SHP/PARTZELAK-A078.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A079/079_HIRILUR_SHP/PARTZELAK-A079.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A080/080_HIRILUR_SHP/PARTZELAK-A080.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A081/081_HIRILUR_SHP/PARTZELAK-A081.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A082/082_HIRILUR_SHP/PARTZELAK-A082.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A083/083_HIRILUR_SHP/PARTZELAK-A083.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A084/084_HIRILUR_SHP/PARTZELAK-A084.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A085/085_HIRILUR_SHP/PARTZELAK-A085.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A086/086_HIRILUR_SHP/PARTZELAK-A086.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A087/087_HIRILUR_SHP/PARTZELAK-A087.shp",
                    "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A088/088_HIRILUR_SHP/PARTZELAK-A088.shp",
                    # "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A098/098_HIRILUR_SHP/PARTZELAK-A098.shp",
                    # "C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Datos/Catastro_referencias_catastrales/unzipped/SHP-A099/099_HIRILUR_SHP/PARTZELAK-A099.shp"
                  ]
for path in paths_shapefiles:
    df = gpd.read_file(path)
    buildings_catastro.append(df)
buildings_catastro = pd.concat(buildings_catastro, ignore_index=True)
buildings_catastro

,REFCAT,Herria,ZCAT,geometry
0,7366011,001,900,"POLYGON ((573290.672 4766396.906, 573290.444 4..."
1,7366002,001,900,"POLYGON ((572958.509 4766049.446, 572967.392 4..."
2,7366019,001,900,"POLYGON ((573325.953 4766413.788, 573313.578 4..."
3,7266055,001,601,"POLYGON ((572839.41 4766433.746, 572837.287 47..."
4,7266054,001,601,"POLYGON ((572789.485 4766468.125, 572780.641 4..."
...,...,...,...,...
58739,7974022,088,601,"POLYGON ((579412.842 4774115.07, 579412.911 47..."
58740,7974024,088,011,"POLYGON ((579508.127 4774273.04, 579507.881 47..."
58741,7974041,088,011,"POLYGON ((579438.914 4774152.12, 579447.301 47..."
58742,7874006,088,900,"POLYGON ((578214.782 4774367.523, 578217.361 4..."


In [9]:
# Rename columns so that they match
buildings_catastro.rename(columns={'REFCAT': 'Referencia'}, inplace=True)
buildings_catastro.rename(columns={'Herria': 'Mun'}, inplace=True)
buildings_catastro = buildings_catastro.drop(columns='ZCAT')

buildings_catastro['Mun'] = buildings_catastro['Mun'].str.lstrip('0')

# str to int64 so that they coincide with the other df
buildings_catastro['Mun'] = pd.to_numeric(buildings_catastro['Mun'], errors='coerce', downcast='integer')
buildings_catastro['Mun'] = buildings_catastro['Mun'].astype(np.int64)
buildings_catastro

,Referencia,Mun,geometry
0,7366011,1,"POLYGON ((573290.672 4766396.906, 573290.444 4..."
1,7366002,1,"POLYGON ((572958.509 4766049.446, 572967.392 4..."
2,7366019,1,"POLYGON ((573325.953 4766413.788, 573313.578 4..."
3,7266055,1,"POLYGON ((572839.41 4766433.746, 572837.287 47..."
4,7266054,1,"POLYGON ((572789.485 4766468.125, 572780.641 4..."
...,...,...,...
58739,7974022,88,"POLYGON ((579412.842 4774115.07, 579412.911 47..."
58740,7974024,88,"POLYGON ((579508.127 4774273.04, 579507.881 47..."
58741,7974041,88,"POLYGON ((579438.914 4774152.12, 579447.301 47..."
58742,7874006,88,"POLYGON ((578214.782 4774367.523, 578217.361 4..."


# Merge both DataFrames, the one with land-uses and the one with geometries. Both have the RC ID number in common.

In [10]:
# When saving, the type sometimes changes

buildings_catastro['Referencia'] = buildings_catastro['Referencia'].astype('int64')

# # List chain to list of ints
# def str_to_int_list(s):
#     return [int(x) for x in s.strip("[] ").split(",")]

def str_to_int_list(s):
    if isinstance(s, list):
        return s  # ya es lista, no hacer nada
    elif isinstance(s, str):
        return [int(x) for x in s.strip("[] ").split(",") if x]
    else:
        return []  # opcional: manejar NaN u otros casos
    
analisis_x_referencia['Superficie Total'] = analisis_x_referencia['Superficie Total'].apply(str_to_int_list)

# Apply function
# analisis_x_referencia['Superficie Total'] = analisis_x_referencia['Superficie Total'].apply(str_to_int_list)

In [11]:
# ANALYSIS 1)
import ast

buildings_1 = pd.merge(buildings_catastro, analisis_x_referencia, on='Referencia', how='inner')
buildings_1 = buildings_1.drop(columns='Mun_y')
buildings_1 = buildings_1.rename(columns={'Mun_x': 'Municipio','Referencia': 'Referencia catastral'})

def safe_eval(x):
    if isinstance(x, list):
        return x  # ya está bien
    elif isinstance(x, str):
        return ast.literal_eval(x)
    else:
        return x  # o [] si prefieres limpiar

buildings_1['Destinos'] = buildings_1['Destinos'].apply(safe_eval)

# import ast
# buildings_1['Destinos'] = buildings_1['Destinos'].apply(lambda x: ast.literal_eval(x))
# # buildings_1 = buildings_1.drop_duplicates()
# buildings_1

In [12]:
# ANALYSIS 2)

# Create a dictionaty
buildings_2 = {'Referencia catastral': buildings_1['Referencia catastral']}

# Obtain all the unique destinations
todos_destinos = set(destino for destinos in buildings_1['Destinos'] for destino in destinos)

# Create a column for each destination and set the area or 0 if doesn't exist
for destino in todos_destinos:
    superficies = []
    for destinos, superficie in zip(buildings_1['Destinos'], buildings_1['Superficie Total']):
        if destino in destinos:
            superficie_index = destinos.index(destino)
            superficies.append(superficie[superficie_index])
        else:
            superficies.append(0)
    buildings_2[destino] = superficies

# Create new df
buildings_2 = pd.DataFrame(buildings_2)

# Add town code and geomatry to buildings_2
buildings_2 = pd.merge(buildings_2, buildings_1, on='Referencia catastral', how='inner')
columns_to_drop = ['Destinos','Porcentajes','Superficie Total']
buildings_2 = buildings_2.drop(columns=columns_to_drop)
buildings_2 = buildings_2.drop_duplicates()

In [14]:
# Now the last change: delete "Referencia catastral" and change the Municipio code with the name of the town

# Bring the town database
towns = pd.read_excel("C:/Users/iazcarateu/Desktop/CSL@Gipuzkoa/Accessibility/1_land_use_cadastral_references/data_towns.xlsx")
towns.head()

# Merge based on "Municipio" and "codigo", maintain just "town"
buildings_2 = pd.merge(buildings_2, towns[['Código', 'Town']], left_on='Municipio', right_on='Código', how='left')

buildings_2.drop(columns='Código', inplace=True)
buildings_2

buildings_2['Descripcion'] = None

# Order columns

orden_columnas = ['Town', 'Referencia catastral', 'Descripcion',
       'geometry', 'Edificabilidad', 'Solar', 'Vivienda',
       'Trastero', 'Garaje Cerrado', 'Agropecuario', 'Local Comercial',
       'Industrial', 'Almacen', 'Deportivo', 'Cultural', 'Educativo',
       'Oficina', 'Hosteleria', 'Obras Urbanizacion', 'Altura menor 1.5 mts',
       'Terraza', 'Accesos y pasillos', 'Jardin', 'Sanidad', 'Deposito',
       'Duplex', 'Plaza Parking', 'Religioso', 'Espectaculos']

buildings_2 = buildings_2[orden_columnas]

# Change to gdf
buildings_2 = gpd.GeoDataFrame(buildings_2)

buildings_2

,Town,Referencia catastral,Descripcion,geometry,Edificabilidad,Solar,Vivienda,Trastero,Garaje Cerrado,Agropecuario,...,Altura menor 1.5 mts,Terraza,Accesos y pasillos,Jardin,Sanidad,Deposito,Duplex,Plaza Parking,Religioso,Espectaculos
0,Abaltzisketa,7366011,None,"POLYGON ((573290.672 4766396.906, 573290.444 4...",0,0,97,0,0,487,...,0,0,0,0,0,0,0,0,0,0
1,Abaltzisketa,7366002,None,"POLYGON ((572958.509 4766049.446, 572967.392 4...",0,0,198,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Abaltzisketa,7366019,None,"POLYGON ((573325.953 4766413.788, 573313.578 4...",0,0,153,118,59,59,...,0,0,0,0,0,0,0,0,0,0
3,Abaltzisketa,7266055,None,"POLYGON ((572839.41 4766433.746, 572837.287 47...",0,756,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Abaltzisketa,7266054,None,"POLYGON ((572789.485 4766468.125, 572780.641 4...",0,1991,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58494,Gaztelu,7974022,None,"POLYGON ((579412.842 4774115.07, 579412.911 47...",0,1389,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
58495,Gaztelu,7974024,None,"POLYGON ((579508.127 4774273.04, 579507.881 47...",0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
58496,Gaztelu,7974041,None,"POLYGON ((579438.914 4774152.12, 579447.301 47...",0,0,0,0,0,0,...,0,0,0,0,0,40,0,0,0,0
58497,Gaztelu,7874006,None,"POLYGON ((578214.782 4774367.523, 578217.361 4...",0,0,97,0,35,0,...,0,0,0,0,0,0,0,0,0,0


In [15]:
buildings_2.to_csv("referencias_catastrales_usos.csv", index=False)

In [23]:
# Change to gdf
buildings_with_section = gpd.GeoDataFrame(buildings_2)

# Save the GeoDataFrame to a shapefile
buildings_with_section.to_file("buildings_with_section/buildings_with_section.shp", index=False)

# buildings_with_section.plot()

C:\Users\iazcarateu\AppData\Local\Temp\ipykernel_16048\3642190131.py:5: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  buildings_with_section.to_file("buildings_with_section/buildings_with_section.shp", index=False)
c:\Users\iazcarateu\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Referencia catastral' to 'Referencia'
  ogr_write(
c:\Users\iazcarateu\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Descripcion' to 'Descripcio'
  ogr_write(
c:\Users\iazcarateu\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'Edificabilidad' to 'Edificabil'
  ogr_write(
c:\Users\iazcarateu\AppData\Local\Programs\Python\Python312\Lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 

# PART 2: Merge with Gipuzkoa's sections and their mean income

I upload sections (bigger areas than RCs) with their respective mean income, so that I can then assign an income to each RC.

Actually I don't assign the mean income, I assign the percentile of income within the same town.

The analysis is not complete because there is economic information about less than half of the towns

In [24]:
# I upload the shapefile of the sections of Gipuzkoa, which I will use to assign a section to each cadastral reference. I will do this by doing a spatial join between the shapefile and the dataframe with the cadastral references.

secciones = gpd.read_file("sections_gipuzkoa/sections_gipuzkoa.shp")
secciones = secciones[secciones['NPRO'] == 'Gipuzkoa']
secciones = secciones[['CUSEC', 'geometry', 'NMUN']]
secciones

,CUSEC,geometry,NMUN
13571,2000101001,"POLYGON ((573368.019 4768463.392, 573369.325 4...",Abaltzisketa
13572,2000201001,"POLYGON ((578062.469 4783728.178, 578060.466 4...",Aduna
13573,2000301001,"POLYGON ((564328.74 4791430.125, 564338.362 47...",Aizarnazabal
13574,2000401001,"POLYGON ((572147.692 4775155.388, 572132.633 4...",Albiztur
13575,2000501001,"POLYGON ((574031.685 4773698.837, 574080.435 4...",Alegia
...,...,...,...
14111,2090301003,"POLYGON ((585338.845 4792840.211, 585354.984 4...",Astigarraga
14112,2090401001,"POLYGON ((571620.111 4768809.437, 571610.801 4...",Baliarrain
14113,2090501001,"POLYGON ((573228.212 4771390.378, 573300.27 47...",Orendain
14114,2090601001,"POLYGON ((569724.081 4769313.986, 569658.075 4...",Altzaga


In [26]:
# I upload the excel file with the income per section, which I will use to assign an income to each cadastral reference. I will do this by merging the dataframe with the sections and the dataframe with the income per section.

renta_x_seccion =  pd.read_excel("sections_gipuzkoa/income_per_section.xlsx", index_col=0)

# Delete towns and districts
renta_x_seccion = renta_x_seccion[renta_x_seccion['Seccion'].str.contains(r'\D*\d{5,}$')]
renta_x_seccion = renta_x_seccion.reset_index(drop=True)

# Save section identifier
renta_x_seccion['Seccion'] = renta_x_seccion['Seccion'].str.extract(r'(\d+)')
renta_x_seccion

,Seccion,Renta neta media por persona
0,2000101001,18052
1,2000201001,16103
2,2000301001,14704
3,2000401001,16229
4,2000501001,14726
...,...,...
540,2090301003,16057
541,2090401001,16428
542,2090501001,16526
543,2090601001,14557


In [27]:
# Merge both DataFrames

df = pd.merge(secciones, renta_x_seccion, left_on='CUSEC', right_on='Seccion', how='left')
df = df.drop(columns='Seccion')
df = df.rename(columns={'CUSEC': 'Seccion', 'Renta neta media por persona ': 'Renta_media'})
df

,Seccion,geometry,NMUN,Renta_media
0,2000101001,"POLYGON ((573368.019 4768463.392, 573369.325 4...",Abaltzisketa,18052
1,2000201001,"POLYGON ((578062.469 4783728.178, 578060.466 4...",Aduna,16103
2,2000301001,"POLYGON ((564328.74 4791430.125, 564338.362 47...",Aizarnazabal,14704
3,2000401001,"POLYGON ((572147.692 4775155.388, 572132.633 4...",Albiztur,16229
4,2000501001,"POLYGON ((574031.685 4773698.837, 574080.435 4...",Alegia,14726
...,...,...,...,...
540,2090301003,"POLYGON ((585338.845 4792840.211, 585354.984 4...",Astigarraga,16057
541,2090401001,"POLYGON ((571620.111 4768809.437, 571610.801 4...",Baliarrain,16428
542,2090501001,"POLYGON ((573228.212 4771390.378, 573300.27 47...",Orendain,16526
543,2090601001,"POLYGON ((569724.081 4769313.986, 569658.075 4...",Altzaga,14557


Assign percentile to each section

In [19]:
# Delete towns with just one section, otherwise error when assigning percentile, obviously

# Count each value's frequency on "Columna"
value_counts = df['NMUN'].value_counts()

# Filter df so that only rows with repeated values appear
df = df[df['NMUN'].isin(value_counts[value_counts > 1].index)]
df = df.reset_index(drop=True)

df['Percentil'] = df.groupby('NMUN')['Renta_media'].transform(
    lambda x: pd.qcut(x, q=[0, 0.25, 0.5, 0.75, 1], labels=[1, 2, 3, 4])
)
df

,Seccion,geometry,NMUN,Renta_media,Percentil
0,2000901001,"POLYGON ((579824.44 4785932.576, 579844.587 47...",Andoain,14819,1
1,2000901002,"POLYGON ((579575.205 4785316.12, 579573.393 47...",Andoain,15807,4
2,2000901003,"POLYGON ((579495.261 4784945.126, 579468.015 4...",Andoain,14905,2
3,2000901004,"POLYGON ((579561.898 4785537.618, 579532.149 4...",Andoain,15192,2
4,2000901005,"POLYGON ((579693.588 4785883.612, 579693.403 4...",Andoain,14725,1
...,...,...,...,...,...
493,2090201012,"POLYGON ((578987.848 4791557.565, 579210.208 4...",Lasarte-Oria,14370,1
494,2090201013,"POLYGON ((579951.006 4792246.518, 579954.386 4...",Lasarte-Oria,17317,4
495,2090301001,"POLYGON ((587093.652 4794882.225, 587080.653 4...",Astigarraga,16956,4
496,2090301002,"POLYGON ((588243.634 4790624.367, 588242.747 4...",Astigarraga,16663,2


Merge income percentile DataFrame to RC DataFrame, assigning a in-town income percentile to each RC.

In [29]:
# Include section to building
buildings_w_section = gpd.sjoin(buildings_2, df, how='left', predicate='within')

# Drop the rows that have not been assigned whichever was the reason. It goes from 45100 to 43503 buildings, hope doesn't affect much.
buildings_w_section = buildings_w_section.dropna(subset=['NMUN'])

buildings_w_section['Percentil'] = buildings_w_section['Percentil'].astype(str)
buildings_w_section

,Town,Referencia catastral,Descripcion,geometry,Edificabilidad,Solar,Vivienda,Trastero,Garaje Cerrado,Agropecuario,...,Deposito,Duplex,Plaza Parking,Religioso,Espectaculos,index_right,Seccion,NMUN,Renta_media,Percentil
1599,Andoain,7985343,None,"POLYGON ((579604.914 4785088.571, 579605.304 4...",0,0,248,124,112,0,...,0,0,0,0,0,9.0,2000901010,Andoain,15277.0,3
1600,Andoain,7985291,None,"POLYGON ((579604.353 4785064.912, 579604.608 4...",0,0,184,30,90,0,...,0,0,0,0,0,9.0,2000901010,Andoain,15277.0,3
1601,Andoain,7985285,None,"POLYGON ((579448.733 4785146.712, 579451.683 4...",0,0,80,0,0,0,...,0,0,0,280,0,1.0,2000901002,Andoain,15807.0,4
1602,Andoain,8086066,None,"POLYGON ((580153.848 4786251.753, 580148.061 4...",0,0,320,0,0,0,...,0,0,0,0,0,7.0,2000901008,Andoain,15368.0,3
1603,Andoain,8086073,None,"POLYGON ((580153.848 4786251.753, 580145.485 4...",0,83,0,0,0,0,...,0,0,0,0,0,7.0,2000901008,Andoain,15368.0,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46152,Astigarraga,8694007,None,"POLYGON ((586589.344 4794034.001, 586595.048 4...",0,0,0,0,0,145,...,0,0,0,0,0,495.0,2090301001,Astigarraga,16956.0,4
46153,Astigarraga,8592128,None,"POLYGON ((585187.1 4792542.262, 585170.809 479...",0,0,3244,5,257,0,...,0,201,469,0,0,497.0,2090301003,Astigarraga,16057.0,1
46154,Astigarraga,8592103,None,"POLYGON ((585032.124 4792095.039, 585049.175 4...",0,0,0,0,0,0,...,0,0,0,0,0,496.0,2090301002,Astigarraga,16663.0,2
46155,Astigarraga,8592266,None,"POLYGON ((584995.603 4792096.148, 584992.453 4...",0,0,5564,832,0,0,...,0,0,944,0,0,496.0,2090301002,Astigarraga,16663.0,2
